# vlti_loader — Tutorial Notebook

This notebook walks through all features of the `vlti_loader` package for loading, inspecting, filtering, plotting, and exporting VLTI interferometric (OIFITS) data.

**Supported instruments:** GRAVITY (K-band), MATISSE (L/M/N-band), PIONIER (H-band)

In [ ]:
# ── User input ─────────────────────────────────────────────────────────────
# Set the path to your OIFITS file (or a directory containing .fits files,
# or a Python list of files from different instruments).


fits_path = ""   # ← change this

# Examples:
#   fits_path = "/data/MATISSE/mystar_LM.fits"
#   fits_path = "/data/GRAVITY/"                              # whole directory
#   fits_path = ["/data/MATISSE/lm.fits", "/data/GRAVITY/k.fits"]  # multi-instrument

## 1 — Load data

In [ ]:
from vlti_loader import Observations
obs = Observations(fits_path)
print(obs)

## 2 — Inspect with `summary()`

In [ ]:
obs.summary()

## 3 — Attribute-style data access

All keys of `obs.data` are also accessible directly as attributes.

In [ ]:
import numpy as np

# These two are equivalent:
vis2_a = obs.data['VIS2']
vis2_b = obs.VIS2
print("Arrays equal:", np.array_equal(vis2_a, vis2_b))

# Other readily available quantities:
print("Wavelength range (µm):", obs.VIS2_waves.min()*1e6, "–", obs.VIS2_waves.max()*1e6)
print("Baseline range   (m) :", obs.B.min(), "–", obs.B.max())
print("Spatial freqs (rad⁻¹):", obs.freqs.min(), "–", obs.freqs.max())
print("u / v (rad⁻¹) sample :", obs.u[:3], obs.v[:3])

# Closure phase (if available)
if 'T3_PHI' in obs.data:
    print("CP range (deg)       :", obs.T3_PHI.min(), "–", obs.T3_PHI.max())

## 4 — Angular resolution table

Prints $\lambda / B$ (in mas) per unique baseline across the observed wavelength range.

In [ ]:
obs.get_effective_resolution()

## 5 — Overview plot: UV coverage + V² + closure phase

In [ ]:
fig = obs.plot(show_uv=True, show=True, v2_ylim=(0,2))


### 5a — Colour by telescope pair (with legend)

Pass `color_by='baseline'` to colour each telescope pair (and each closure-phase triangle)
with a distinct colour and show a legend with the telescope names instead of the wavelength colour bar.

In [ ]:
# Same axes as plot(), but each baseline/triangle gets its own colour + a legend
fig = obs.plot(show_uv=True, color_by='baseline', v2_ylim=(0, 2))

## 5b — Plot V² and CP vs wavelength

`plot_wavelength` shows the same data as `plot` but with wavelength on the x-axis.
Lines are colour-coded by baseline length (V²) or maximum baseline (CP).

In [ ]:
fig = obs.plot_wavelength(v2_ylim=(0, 1.4), cp_ylim=(-30, 30))

## 6 — Filter data

`filter_data` accepts a `(min, max)` tuple, or a list of tuples for multiple disjoint windows.  
Filters are combined with AND across parameters; multiple windows within one parameter are combined with OR.  
The operation updates `obs.data` in-place — use `reset_data()` to undo it.

In [ ]:
help(obs.filter_data)

In [ ]:
# Keep PIONIER (H-band), GRAVITY (K-band), and MATISSE L-band; discard very noisy points
obs.filter_data(
    wave=[(1.5e-6, 1.8e-6), (2.0e-6, 2.4e-6), (3.2e-6, 3.9e-6)],
    vis2_err=(0, 0.15),
    cp_err=(0, 20),
)
obs.summary()

In [ ]:
fig = obs.plot(show_uv=True, v2_ylim=(0, 1.2), cp_ylim=(-30, 30))

## 6b — Flag V² baselines and T3 triangles by name

`flag_v2` removes V² data for specific telescope pairs (T3 unchanged).
`flag_t3` removes T3 closure-phase data for specific triangles (V² unchanged).
Both accept a telescope pair/triplet string *or* a single telescope name.
Use `reset_data()` to undo.

In [ ]:
# flag_v2: remove a single baseline pair (T3 data untouched)
obs.flag_v2(baselines='AT1-AT2')
obs.plot()

In [ ]:
# flag_v2: remove all V² baselines involving a specific telescope
obs.reset_data()
obs.flag_v2(telescopes='AT1')
obs.plot()

In [ ]:
# flag_t3: remove a specific closure-phase triangle by triplet name (V² untouched)
obs.reset_data()
obs.flag_t3(triangles='AT1-AT2-AT3')
obs.plot()

In [ ]:
# flag_t3: remove all T3 triangles involving a specific telescope
obs.reset_data()
obs.flag_t3(telescopes='AT1')
obs.plot()

## 7 — Spectral binning

Combine every `n` spectral channels to boost SNR at the cost of spectral resolution.  
Errors are propagated in quadrature.

In [ ]:
obs.reset_data()   # start from the full unfiltered dataset
obs.filter_data(
    wave=[(1.5e-6, 1.8e-6), (2.0e-6, 2.4e-6), (3.2e-6, 3.9e-6)],
    vis2_err=(0, 0.15),
    cp_err=(0, 20),
)
obs.bin_spectral_channels(bin_size=5)
obs.summary()
fig = obs.plot(show_uv=True, v2_ylim=(0, 1.2))

## 7b — Fit a uniform-disk model

`fit_uniform_disk` fits a UD visibility model
to the V² data using χ² minimisation.
The result dictionary also contains `model_vis2` which you can overlay on `plot`.

In [ ]:
result = obs.fit_uniform_disk(theta_init=1.0)
print(f"Diameter: {result['theta_mas']:.3f} ± {result['theta_err_mas']:.3f} mas")
print(f"Reduced χ²: {result['chi2_red']:.2f}")

In [ ]:
# Overlay the best-fit model on the spatial-frequency plot
fig = obs.plot(model_vis2=result['model_vis2'], v2_ylim=(0, 1.2))

## 8 — Reset to original data

Restores `obs.data` to the original state loaded from disk — no file re-read required.

In [ ]:
obs.reset_data()
obs.summary()

## 9 — Export to OIFITS

Write the current (possibly filtered/binned) data to a valid OIFITS 2 file,
ready for model-fitting codes such as **PMOIRED**, **CANDID**, or **ASPRO2**.

> **Note:** target coordinates, telescope positions, and timestamps are not stored
> in the `vlti_loader` data model and will be written as placeholder zeros.

In [ ]:
import os

base_dir = os.path.dirname(fits_path) if not os.path.isdir(fits_path) else fits_path
output_path = os.path.join(base_dir, "filtered_output.fits")

# Apply a quality filter across all instrument bands, then export
obs.filter_data(
    wave=[(1.5e-6, 1.8e-6), (2.0e-6, 2.4e-6), (3.2e-6, 3.9e-6)],
    vis2_err=(0, 0.15),
)
print("Saved to:", output_path)
obs.export_oifits(output_path)